# Microsoft Agent Framework — Azure OpenAI (Responses API)

ဤကုဒ်နမူနာတွင် **Microsoft Agent Framework (MAF)** ကို အသုံးပြု၍ **Azure OpenAI** အား အားပံ့ပိုးတဲ့ ရိုးရှင်းသော agent တစ်ခုကို **Responses API** ဖြင့် ဖန်တီးပုံကို သင်ကြားပါမည်။

> **လွှဲပြောင်းမှု မှတ်ချက်** - ဤနမူနာကို ယခင်တွင် Semantic Kernel နှင့် GitHub Models အသုံးပြုခဲ့သည်။ ယခု Microsoft Agent Framework သို့လွှဲပြောင်းပြီး GitHub Models (အသုံးမပြုတော့၊ ၂၀၂၆ ခုနှစ် ဇူလိုင်တွင် ပိတ်သိမ်းမည်) ကို Azure OpenAI ဖြင့် အစားအစားထိုးထားပြီး Responses API ကို ထောက်ပံ့သည်။ MAF မှာ `OpenAIChatClient` သည် Azure OpenAI ၏ တည်ငြိမ်သော `/openai/v1/` endpoint ကို ဦးတည်ပြီး Responses API ကို ပုံမှန်အဖြစ် အသုံးပြုသည်။

ဤနမူနာ၏ရည်ရွယ်ချက်မှာ agentic ပုံစံမျိုးစုံ အကောင်အထည်ဖော်ရာတွင် နောက်ပိုင်းတွင် အသုံးပြုမည့် အခြားကုဒ်နမူနာများတွင် လိုက်နာဆောင်ရွက်ရမည့် လုပ်ဆောင်ချက်များကို ဖေါ်ပြရန်ဖြစ်သည်။


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## လိုအပ်သော Python ပက်ကေ့ချ်များ ကို တင်သွင်းခြင်း


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ကိရိယာတစ်ခု တိမ်းမှတ်ခြင်း

Microsoft Agent Framework တွင်၊ **ကိရိယာ** ဆိုသည်မှာ agent သည် ခေါ်နိုင်သော `@tool` ဖြင့် အလှည့်အပြောင်းသတ်မှတ်ထားသော ရိုးရှင်းသော Python လုပ်ဆောင်ချက်တစ်ခု ဖြစ်သည်။ အောက်တွင် ကျွန်ုပ်တို့သည် ယခင် မည်သည့်နေရာကိုမှ ထပ်မံမသွားရန်နှင့် ပြန်လည်ရွေးချယ်မရသော အနားယူခရီးစဉ်တိမ်းမှတ်တစ်ခုကို သတ်မှတ်ပေးထားသည်။


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Agent ဖန်တီးခြင်း

 ဒီမှာ၊ `TravelAgent` ဟု အမည်ပေးထားသော Agent ကို ဖန်တီးပါသည်။

 ဥပမာအနေဖြင့် အလွန်ရိုးရှင်းသောညွှန်ကြားချက်များကို အသုံးပြုထားပါသည်။ Agent ၏အပြုအမူ မည်သို့ပြောင်းလဲသွားသည်ကို ကြည့်ရှုနိုင်ရန် ဤညွှန်ကြားချက်များကို ပြင်ဆင်ရန် လွတ်လပ်ပါသည်။


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ကိုယ်စားလှယ်ကို အသုံးပြုခြင်း

ယခု ကျွန်ုပ်တို့သည် ကိုယ်စားလှယ်ကို အသုံးပြုနိုင်ပါပြီ။ ကိုယ်စားလှယ်သည် ပြောဆိုဆက်သွယ်မှုကို မောင်းနှင်နိုင်ရန် `AgentSession` တစ်ခု ဖန်တီးပြီး၊ ထို့နောက် `user_inputs` နှစ်ခုကို ပို့ပေးသည်။ ပထမတစ်ခုသည် ခရီးသွားမှုကို မေးမြန်းပြီး၊ ဒုတိယတစ်ခုသည် အသုံးပြုသူက အကြံပြုချက်ကို မနှစ်မြို့သဖြင့် နောက်တစ်ခုကို မေးမြန်းသည်။ ကိုယ်စားလှယ်သည် session သမိုင်းဖြင့်အတူ `get_random_destination` ကိရိယာကို အသုံးပြု၍ တုံ့ပြန်သည်။

သင်သည် မက်ဆေ့ခ်ျများကို ပြောင်းလဲ၍ ကိုယ်စားလှယ်၏ တုံ့ပြန်မှု ကွဲပြားမှုကို ကြည့်ရှုနိုင်သည်။ တုံ့ပြန်မှုများသည် **token-by-token** ဖြင့် **စီးဆင်း ထုတ်လွင့်** လုပ်သည်။


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
